In [ ]:
#!/usr/bin/env python3
"""
XLM-RoBERTa-large + CRF with hard BIO transition constraints
Optimized Hyperparameters for Contextual Learning & Regularization.
INCLUDES LIVE DIAGNOSTICS, ANOMALY TRACKING, and EXTERNAL SPLIT-ACCURACY EVAL.
External evaluation (every EVAL_EVERY_STEPS + end of every epoch) is for monitoring only —
it does NOT affect the training loop, early-stopping, or checkpoint selection.
Supports resume from existing checkpoints.
Saves a full checkpoint after every epoch (like the BiLSTM script).
"""
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import gc
import json
import re
import time
import torch
import subprocess
from collections import defaultdict, Counter
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from datasets import Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    DataCollatorForTokenClassification,
    get_linear_schedule_with_warmup
)
from torchcrf import CRF
import shutil

# ------------------------------------------------------------------
# Optimized Hyperparameters
# ------------------------------------------------------------------
MODEL_PATH = "xlm-roberta-large"
OUTPUT_DIR = "./xlm_roberta_large_crfV4"
CHECKPOINT_DIR = "./xlm_roberta_large_checkpointV4"
DATA_CACHE_DIR = "./xlm_roberta_large_cacheV4"
DIAGNOSTIC_LOG = "training_diagnostics.log"
METRICS_LOG = os.path.join(OUTPUT_DIR, "training_metrics.log")
METRICS_JSONL = os.path.join(OUTPUT_DIR, "metrics.jsonl")

TRAIN_FILE = "data2/train_cleaned.jsonl"
VAL_FILE = "data2/validation_cleaned.jsonl"
TEST_FILE = "data2/test_cleaned.jsonl"
# External monitoring set (TXT or JSONL). Does NOT affect training.
EXTERNAL_TEST_FILE = "data2/address_dataset.jsonl"  # change to your real path

EVAL_EVERY_STEPS = 2000
EPOCHS = 5
LEARNING_RATE = 8e-6
WEIGHT_DECAY = 0.05
BATCH_SIZE = 16
ACCUMULATION_STEPS = 8
MAX_LEN = 128
SAVE_STEPS = 300  # intermediate safety saves (still kept)

ALL_FIELDS = [
    "flat", "floor", "building_name", "block", "phase",
    "estate_name", "village_name", "building_number",
    "street_name", "sub_district", "district", "region"
]

# ------------------------------------------------------------------
# Model
# ------------------------------------------------------------------
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list
        self.bert = AutoModel.from_pretrained(
            model_name_or_path, config=config, ignore_mismatched_sizes=True
        )
        self.dropout = torch.nn.Dropout(0.2)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)
        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)
        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}
        o_id = label2id["O"]
        self.crf.transitions.data[o_id, o_id] = 0.0
        for tag in label_list:
            if tag.startswith("B-"):
                self.crf.transitions.data[o_id, label2id[tag]] = 0.0
        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0
        for tag in label_list:
            if tag == "O":
                continue
            tid = label2id[tag]
            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                self.crf.transitions.data[tid, o_id] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
                self.crf.start_transitions.data[tid] = 0.0
                self.crf.end_transitions.data[tid] = 0.0
            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                self.crf.transitions.data[tid, o_id] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
                self.crf.end_transitions.data[tid] = 0.0
        self.crf.transitions.requires_grad_(False)
        self.crf.start_transitions.requires_grad_(False)
        self.crf.end_transitions.requires_grad_(False)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)
        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            loss = -self.crf(emissions, tags=safe_labels, mask=mask, reduction="mean")
            return loss
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions

    def save_pretrained(self, save_directory):
        os.makedirs(save_directory, exist_ok=True)
        self.config.save_pretrained(save_directory)
        torch.save(self.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))


def get_emptiest_gpu_safely():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu",
             "--format=csv,nounits,noheader"],
            encoding="utf-8"
        )
        best_id, max_free_mb = -1, 0
        fallback_id, fallback_max_mb = 0, 0
        for line in result.strip().split("\n"):
            parts = line.split(", ")
            gpu_id = int(parts[0])
            free_memory = int(parts[1])
            gpu_util = int(parts[2])
            if free_memory > fallback_max_mb:
                fallback_max_mb = free_memory
                fallback_id = gpu_id
            if gpu_util < 30 and free_memory > max_free_mb:
                max_free_mb = free_memory
                best_id = gpu_id
        if best_id != -1:
            print(f"--> Selected GPU {best_id} with {max_free_mb} MB free VRAM.\n")
            return torch.device(f"cuda:{best_id}")
        else:
            return torch.device(f"cuda:{fallback_id}")
    except Exception:
        return torch.device("cuda:0")


# ------------------------------------------------------------------
# Label reconstruction (training data)
# ------------------------------------------------------------------
def reconstruct_char_labels(input_text, output_dict):
    char_labels = ["O"] * len(input_text)
    field_to_tag = {
        "flat": "UNIT", "floor": "FLOOR", "block": "BLOCK", "phase": "PHASE",
        "building_name": "BUILDING_NAME", "estate_name": "ESTATE_NAME",
        "street_name": "STREET_NAME", "sub_district": "SUB_DISTRICT",
        "district": "DISTRICT", "region": "REGION",
        "village_name": "VILLAGE_NAME", "building_number": "BUILDING_NUMBER"
    }
    flat_targets = {}
    if "line1" in output_dict:
        flat_targets.update(output_dict.get("line1", {}))
        flat_targets.update(output_dict.get("line2", {}))
    else:
        flat_targets = output_dict
    items_to_process = []
    for field, tag in field_to_tag.items():
        val = flat_targets.get(field, "")
        if not val:
            continue
        search_terms = val.split(" / ") if " / " in val else [val]
        for term in search_terms:
            if term:
                items_to_process.append((field, tag, term))
    items_to_process.sort(key=lambda x: len(x[2]), reverse=True)
    lower_input = input_text.lower()
    for field, tag, term in items_to_process:
        start_idx = 0
        lower_term = term.lower()
        while True:
            idx = lower_input.find(lower_term, start_idx)
            if idx == -1:
                break
            is_already_tagged = any(
                char_labels[i] != "O" for i in range(idx, idx + len(term))
            )
            if not is_already_tagged:
                char_labels[idx] = f"B-{tag}"
                for i in range(idx + 1, idx + len(term)):
                    if i < len(char_labels):
                        char_labels[i] = f"I-{tag}"
                break
            else:
                start_idx = idx + 1
    return char_labels


def parse_and_tokenize(tokenizer):
    print("📂 Parsing datasets and reconstructing NER labels...")
    unique_labels = {"O"}
    tag_list = [
        "UNIT", "FLOOR", "BUILDING_NAME", "ESTATE_NAME", "STREET_NAME",
        "SUB_DISTRICT", "DISTRICT", "REGION", "VILLAGE_NAME", "BUILDING_NUMBER",
        "BLOCK", "PHASE"
    ]
    for tag in tag_list:
        unique_labels.add(f"B-{tag}")
        unique_labels.add(f"I-{tag}")
    label_list = sorted(list(unique_labels))
    label_to_id = {l: i for i, l in enumerate(label_list)}
    datasets_dict = {}
    for file_path in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
        if not os.path.exists(file_path):
            continue
        texts, labels_list = [], []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in tqdm(f, desc=f"Reading {file_path}"):
                line = line.strip()
                if not line:
                    continue
                data = json.loads(line)
                input_text = data.get("input", "")
                output_dict = data.get("output", {})
                char_labels = reconstruct_char_labels(input_text, output_dict)
                texts.append(input_text)
                labels_list.append(char_labels)
        datasets_dict[file_path] = Dataset.from_dict({"text": texts, "char_labels": labels_list})
    os.makedirs(DATA_CACHE_DIR, exist_ok=True)
    with open(os.path.join(DATA_CACHE_DIR, "label_map.json"), "w") as f:
        json.dump({"label_list": label_list, "label_to_id": label_to_id}, f)

    def align_labels(examples):
        tokenized = tokenizer(
            examples["text"], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True
        )
        labels = []
        for i, offsets in enumerate(tokenized["offset_mapping"]):
            char_labels = examples["char_labels"][i]
            token_labels = []
            for start_offset, end_offset in offsets:
                if start_offset == end_offset:
                    token_labels.append(-100)
                else:
                    tag = "O"
                    for char_idx in range(start_offset, end_offset):
                        if char_idx < len(char_labels) and char_labels[char_idx] != "O":
                            tag = char_labels[char_idx]
                            break
                    token_labels.append(label_to_id[tag])
            # BIO sanitization
            sanitized_labels = []
            prev_tag = "O"
            for label_id in token_labels:
                if label_id == -100:
                    sanitized_labels.append(-100)
                    continue
                current_tag = label_list[label_id]
                if current_tag.startswith("I-"):
                    entity_type = current_tag[2:]
                    if prev_tag not in [f"B-{entity_type}", f"I-{entity_type}"]:
                        current_tag = f"B-{entity_type}"
                        label_id = label_to_id[current_tag]
                sanitized_labels.append(label_id)
                prev_tag = current_tag
            labels.append(sanitized_labels)
        tokenized["labels"] = labels
        tokenized.pop("offset_mapping")
        return tokenized

    print("⏳ Tokenizing datasets...")
    train_ds = datasets_dict[TRAIN_FILE].map(
        align_labels, batched=True, remove_columns=["text", "char_labels"]
    )
    val_ds = datasets_dict.get(VAL_FILE)
    if val_ds:
        val_ds = val_ds.map(align_labels, batched=True, remove_columns=["text", "char_labels"])
    print(f"💾 Caching tokenized datasets to {DATA_CACHE_DIR}...")
    train_ds.save_to_disk(os.path.join(DATA_CACHE_DIR, "train"))
    if val_ds:
        val_ds.save_to_disk(os.path.join(DATA_CACHE_DIR, "val"))
    return train_ds, val_ds, label_list, label_to_id


def save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, best_val_loss, global_step):
    """Atomic checkpoint for resume."""
    print(f"\n💾 Saving resume checkpoint at Epoch {epoch+1}, Step {step} (gstep={global_step})...")
    temp_dir = f"{CHECKPOINT_DIR}_tmp"
    os.makedirs(temp_dir, exist_ok=True)
    model.save_pretrained(temp_dir)
    tokenizer.save_pretrained(temp_dir)
    state = {
        "epoch": epoch,
        "step": step,
        "global_step": global_step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_loss": best_val_loss
    }
    torch.save(state, os.path.join(temp_dir, "training_state.pt"))
    try:
        os.replace(temp_dir, CHECKPOINT_DIR)
    except OSError:
        if os.path.exists(CHECKPOINT_DIR):
            shutil.rmtree(CHECKPOINT_DIR)
        os.rename(temp_dir, CHECKPOINT_DIR)


def save_epoch_checkpoint(model, tokenizer, optimizer, scheduler, epoch, best_val_loss, global_step, avg_train_loss):
    """Full epoch checkpoint (like BiLSTM)."""
    epoch_dir = os.path.join(OUTPUT_DIR, f"checkpoint_epoch_{epoch+1:02d}")
    os.makedirs(epoch_dir, exist_ok=True)
    model.save_pretrained(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)
    state = {
        "epoch": epoch,
        "global_step": global_step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "train_loss": avg_train_loss,
    }
    torch.save(state, os.path.join(epoch_dir, "training_state.pt"))
    print(f"💾 Saved full epoch checkpoint → {epoch_dir}")


def log_diagnostic(tokenizer, input_ids, labels, preds, id2label, metric_val, epoch, step, context):
    with open(DIAGNOSTIC_LOG, "a", encoding="utf-8") as f:
        f.write(f"\n[{context}] Epoch {epoch+1} | Step {step} | Metric/Loss: {metric_val:.4f}\n")
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
        true_ids = labels[0].tolist()
        pred_ids = preds[0]
        f.write(f"{'TOKEN':<20} | {'TRUE LABEL':<20} | {'PRED LABEL':<20}\n")
        f.write("-" * 65 + "\n")
        pred_idx = 0
        for tok, t_id in zip(tokens, true_ids):
            if tok in [tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token]:
                if tok != tokenizer.pad_token:
                    pred_idx += 1
                continue
            if t_id == -100:
                pred_idx += 1
                continue
            true_tag = id2label.get(t_id, "O")
            p_id = pred_ids[pred_idx] if pred_idx < len(pred_ids) else 0
            pred_tag = id2label.get(p_id, "O")
            pred_idx += 1
            marker = "❌" if true_tag != pred_tag else "✅"
            f.write(f"{tok:<20} | {true_tag:<20} | {pred_tag:<20} {marker}\n")
        f.write("=" * 65 + "\n")


# ------------------------------------------------------------------
# External evaluation helpers (monitoring only)
# ------------------------------------------------------------------
def normalize_for_eval(text):
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())


def content_covered(gt_text, address_norm):
    gt_norm = normalize_for_eval(gt_text)
    if not gt_norm:
        return True
    gt_cnt = Counter(gt_norm)
    addr_cnt = Counter(address_norm)
    return all(addr_cnt[c] >= cnt for c, cnt in gt_cnt.items())


def load_external_test(path):
    """Load TXT (input | line1 | line2) or JSONL. Returns list of dicts."""
    if not os.path.exists(path):
        print(f"⚠️ External test file not found: {path} — skipping external eval.")
        return []
    data = []
    is_jsonl = path.lower().endswith((".jsonl", ".json"))
    with open(path, "r", encoding="utf-8") as f:
        for line_no, raw in enumerate(f, 1):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if is_jsonl:
                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    continue
                inp = item.get("input", "").strip()
                out = item.get("output", {})
                if isinstance(out.get("line1"), dict):
                    def tags_to_str(d):
                        if not isinstance(d, dict):
                            return str(d)
                        return "".join(str(d[k]) for k in ALL_FIELDS if k in d and d[k])
                    gt_l1 = tags_to_str(out.get("line1", {}))
                    gt_l2 = tags_to_str(out.get("line2", {}))
                else:
                    gt_l1 = str(out.get("line1", "")).strip()
                    gt_l2 = str(out.get("line2", "")).strip()
            else:
                if " | " in line:
                    parts = [p.strip() for p in line.split(" | ")]
                else:
                    parts = [p.strip() for p in line.split("|")]
                if len(parts) < 2:
                    continue
                inp = parts[0]
                gt_l1 = parts[1] if len(parts) > 1 else ""
                gt_l2 = parts[2] if len(parts) > 2 else ""
            data.append({"input": inp, "line1": gt_l1, "line2": gt_l2})
    print(f"📂 External test loaded: {len(data)} samples from {path}")
    return data


def split_address_from_entities(extracted_labels, original_input, parsed_entities):
    """Hierarchical split (no reorder). Returns line1, line2."""
    line1_keys = set()
    logic_keys_used = set()          # <-- FIXED: was undefined

    if "flat" in extracted_labels:
        line1_keys.add("flat")
        logic_keys_used.add("flat")
    if "floor" in extracted_labels:
        line1_keys.add("floor")
        logic_keys_used.add("floor")

    has_bldg = "building_name" in extracted_labels
    has_block = "block" in extracted_labels
    has_est = "estate_name" in extracted_labels
    has_phase = "phase" in extracted_labels
    has_vill = "village_name" in extracted_labels
    has_street = "street_name" in extracted_labels
    has_bldg_no = "building_number" in extracted_labels

    if has_block:
        line1_keys.add("block")
        logic_keys_used.add("block")
        if has_bldg and has_est:
            line1_keys.add("building_name")
            logic_keys_used.add("building_name")
        elif has_bldg and not has_est:
            # FIX: Properly add building_name if it exists without an estate
            line1_keys.add("building_name")
            logic_keys_used.add("building_name")
    elif has_bldg:
        line1_keys.add("building_name")
        logic_keys_used.add("building_name")
    elif has_est:
        line1_keys.add("estate_name")
        logic_keys_used.add("estate_name")
        if has_phase:
            line1_keys.add("phase")
            logic_keys_used.add("phase")
    elif has_vill:
        line1_keys.add("village_name")
        logic_keys_used.add("village_name")
        if has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")
    elif has_street:
        line1_keys.add("street_name")
        logic_keys_used.add("street_name")
        if has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")
    elif has_bldg_no:
        line1_keys.add("building_number")
        logic_keys_used.add("building_number")

    is_chinese = any("\u4e00" <= c <= "\u9fff" for c in original_input)
    mapping = {
        "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
        "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
        "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
        "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
        "DISTRICT": "district", "REGION": "region"
    }
    token_groups = []
    for entity in parsed_entities:
        mapped = mapping.get(entity["entity_group"], entity["entity_group"].lower())
        if mapped in line1_keys:
            token_groups.append("micro")
        elif mapped not in ("o", "O"):
            token_groups.append("macro")
        else:
            token_groups.append("O")
    resolved = []
    last = "micro" if not is_chinese else "macro"
    for tg in token_groups:
        if tg != "O":
            last = tg
            resolved.append(tg)
        else:
            resolved.append(last)
    micro_segs, macro_segs = [], []
    cur_m, cur_m_tag = [], None
    cur_M, cur_M_tag = [], None
    for entity, group in zip(parsed_entities, resolved):
        tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
        word = entity["word"]
        if group == "micro":
            if tag not in ("o", "O"):
                if cur_m_tag != tag:
                    if cur_m:
                        micro_segs.append(cur_m)
                    cur_m, cur_m_tag = [word], tag
                else:
                    cur_m.append(word)
            else:
                if cur_m:
                    cur_m.append(word)
                else:
                    cur_m, cur_m_tag = [word], "o"
        else:
            if tag not in ("o", "O"):
                if cur_M_tag != tag:
                    if cur_M:
                        macro_segs.append(cur_M)
                    cur_M, cur_M_tag = [word], tag
                else:
                    cur_M.append(word)
            else:
                if cur_M:
                    cur_M.append(word)
                else:
                    cur_M, cur_M_tag = [word], "o"
    if cur_m:
        micro_segs.append(cur_m)
    if cur_M:
        macro_segs.append(cur_M)

    def clean(s):
        s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
        s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
        return re.sub(r"\s{2,}", " ", s).strip()

    micro_str = clean("".join("".join(w) for w in micro_segs))
    macro_str = clean("".join("".join(w) for w in macro_segs))
    line1 = macro_str if is_chinese else micro_str
    line2 = micro_str if is_chinese else macro_str
    return line1, line2


def run_external_split_eval(model, tokenizer, device, external_data, label_list, batch_size=32):
    """
    Run current model on external set and compute Line1 / Line2 / Full string accuracy.
    Returns dict of metrics. Does NOT touch gradients.
    """
    if not external_data:
        return None
    model.eval()
    mapping = {
        "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
        "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
        "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
        "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
        "DISTRICT": "district", "REGION": "region"
    }
    l1_ok = l2_ok = full_ok = 0
    valid = 0
    texts = [item["input"] for item in external_data]
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded = tokenizer(
                batch_texts, padding=True, truncation=True, max_length=MAX_LEN,
                return_offsets_mapping=True, return_tensors="pt"
            )
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)
            offsets_batch = encoded["offset_mapping"].cpu().numpy()
            preds_batch, _ = model(input_ids=input_ids, attention_mask=attention_mask)
            for b_idx, address_str in enumerate(batch_texts):
                gt_l1 = external_data[i + b_idx]["line1"]
                gt_l2 = external_data[i + b_idx]["line2"]
                norm_addr = normalize_for_eval(address_str)
                if not content_covered(gt_l1, norm_addr) or not content_covered(gt_l2, norm_addr):
                    continue  # skip corrupted
                valid += 1
                prediction_ids = preds_batch[b_idx]
                offsets = offsets_batch[b_idx]
                char_tags = ["O"] * len(address_str)
                for j, tag_id in enumerate(prediction_ids):
                    start, end = offsets[j]
                    if start == end:
                        continue
                    tag = label_list[tag_id]
                    # FIX: Bound 'end' to prevent IndexError from SentencePiece offsets
                    for c in range(start, min(end, len(char_tags))):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing = match.group(2)
                    start_idx = match.start(1)
                    tag = char_tags[start_idx] if start_idx < len(char_tags) else "O"
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing
                    })
                components = defaultdict(list)
                for ent in parsed_entities:
                    if ent["entity_group"] != "O":
                        components[ent["entity_group"]].append(ent["word"])
                extracted_raw = {k: "".join(v) for k, v in components.items()}
                extracted = {mapping.get(k, k.lower()): v for k, v in extracted_raw.items()}
                pred_l1, pred_l2 = split_address_from_entities(
                    extracted, address_str, parsed_entities
                )
                if normalize_for_eval(pred_l1) == normalize_for_eval(gt_l1):
                    l1_ok += 1
                if normalize_for_eval(pred_l2) == normalize_for_eval(gt_l2):
                    l2_ok += 1
                if (normalize_for_eval(pred_l1) == normalize_for_eval(gt_l1) and
                        normalize_for_eval(pred_l2) == normalize_for_eval(gt_l2)):
                    full_ok += 1
    if valid == 0:
        return None
    return {
        "valid": valid,
        "line1_acc": l1_ok / valid * 100,
        "line2_acc": l2_ok / valid * 100,
        "full_acc": full_ok / valid * 100,
        "l1_ok": l1_ok,
        "l2_ok": l2_ok,
        "full_ok": full_ok
    }


def _write_epoch_metrics(log_path, jsonl_path, epoch, train_loss, val_loss, val_em,
                         ext_metrics, was_best, best_val_loss, global_step):
    """Clean BiLSTM-style metrics block + JSONL."""
    lines = []
    lines.append(f"{'─'*90}")
    lines.append(f"Epoch {epoch:02d}  (global_step={global_step})")
    lines.append(f"{'─'*90}")
    lines.append(f" Train loss          : {train_loss:.6f}" if train_loss is not None else " Train loss          : n/a")
    if val_loss is not None:
        lines.append(f" Val loss            : {val_loss:.6f}")
        lines.append(f" Val Exact Match     : {val_em*100:.2f}%")
    else:
        lines.append(f" Val loss            : n/a")
    if ext_metrics is not None:
        lines.append(f" External Line1 Acc  : {ext_metrics['line1_acc']:.2f}%")
        lines.append(f" External Line2 Acc  : {ext_metrics['line2_acc']:.2f}%")
        lines.append(f" External Full Acc   : {ext_metrics['full_acc']:.2f}%")
        lines.append(f" (external valid)    : {ext_metrics['valid']}")
    lines.append(f" Marked as best?     : {'YES ★' if was_best else 'no'}")
    lines.append(f" Running best val_loss: {best_val_loss:.6f}")
    lines.append(f" Checkpoint          : checkpoint_epoch_{epoch:02d}/")
    lines.append("")
    block = "\n".join(lines)
    print(block)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(block)

    record = {
        "epoch": epoch,
        "global_step": global_step,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_em": val_em,
        "ext_line1_acc": ext_metrics["line1_acc"] if ext_metrics else None,
        "ext_line2_acc": ext_metrics["line2_acc"] if ext_metrics else None,
        "ext_full_acc": ext_metrics["full_acc"] if ext_metrics else None,
        "ext_valid": ext_metrics["valid"] if ext_metrics else None,
        "was_best": was_best,
        "best_val_loss": best_val_loss,
        "checkpoint": f"checkpoint_epoch_{epoch:02d}"
    }
    with open(jsonl_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# ------------------------------------------------------------------
# Main training loop
# ------------------------------------------------------------------
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(DIAGNOSTIC_LOG, "w", encoding="utf-8") as f:
        f.write("🚀 Starting New Diagnostic Log Run\n" + "=" * 40 + "\n")

    # init metrics files
    if not os.path.exists(METRICS_LOG):
        with open(METRICS_LOG, "w", encoding="utf-8") as f:
            f.write("=" * 90 + "\n")
            f.write("XLM-RoBERTa-large + CRF Training Metrics Report\n")
            f.write("Use this file + metrics.jsonl to decide which checkpoint_epoch_XX is best.\n")
            f.write("=" * 90 + "\n\n")
    if not os.path.exists(METRICS_JSONL):
        open(METRICS_JSONL, "w").close()

    device = get_emptiest_gpu_safely()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    if os.path.exists(os.path.join(DATA_CACHE_DIR, "train")):
        train_ds = load_from_disk(os.path.join(DATA_CACHE_DIR, "train"))
        val_ds = load_from_disk(os.path.join(DATA_CACHE_DIR, "val")) if os.path.exists(os.path.join(DATA_CACHE_DIR, "val")) else None
        with open(os.path.join(DATA_CACHE_DIR, "label_map.json"), "r") as f:
            label_data = json.load(f)
            label_list = label_data["label_list"]
            label_to_id = {k: int(v) for k, v in label_data["label_to_id"].items()}
    else:
        train_ds, val_ds, label_list, label_to_id = parse_and_tokenize(tokenizer)

    id2label = {int(v): k for k, v in label_to_id.items()}

    # Load external monitoring set once
    external_data = load_external_test(EXTERNAL_TEST_FILE)

    print("⏳ Initialising XLM-RoBERTa-large + constrained CRF...")
    resume_from_checkpoint = os.path.exists(os.path.join(CHECKPOINT_DIR, "training_state.pt"))
    config = AutoConfig.from_pretrained(
        MODEL_PATH, num_labels=len(label_list), id2label=id2label, label2id=label_to_id
    )
    model = BertCRFForTokenClassification(config, MODEL_PATH, label_list)

    if resume_from_checkpoint:
        weights_path = os.path.join(CHECKPOINT_DIR, "pytorch_model.bin")
        if os.path.exists(weights_path):
            print("🔄 Loading weights from checkpoint...")
            model.load_state_dict(torch.load(weights_path, map_location=device))
            model._set_bio_constraints(label_list)

    model.to(device)

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    train_loader = DataLoader(
        train_ds, shuffle=True, batch_size=BATCH_SIZE,
        collate_fn=data_collator, num_workers=4, pin_memory=True
    )
    val_loader = None
    if val_ds:
        val_loader = DataLoader(
            val_ds, batch_size=BATCH_SIZE * 2,
            collate_fn=data_collator, num_workers=4, pin_memory=True
        )

    torch.cuda.empty_cache()
    gc.collect()

    # Optimizer groups
    roberta_params, crf_classifier_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "crf" in name or "classifier" in name:
            crf_classifier_params.append(param)
        else:
            roberta_params.append(param)
    optimizer = AdamW([
        {"params": roberta_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": crf_classifier_params, "lr": 1e-3, "weight_decay": 0.0}
    ])
    total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.2), num_training_steps=total_steps
    )

    start_epoch = 0
    start_step = 0
    best_val_loss = float("inf")
    global_step = 0  # counts optimizer steps (after accumulation)

    if resume_from_checkpoint:
        state = torch.load(os.path.join(CHECKPOINT_DIR, "training_state.pt"), map_location=device)
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        start_epoch = state["epoch"]
        start_step = state["step"]
        best_val_loss = state.get("best_val_loss", float("inf"))
        global_step = state.get("global_step", 0)
        print(f"⏩ Resuming from Epoch {start_epoch+1}, Step {start_step} (global_step={global_step})...")

    print("\n🚀 Training XLM-RoBERTa-large + constrained CRF...")
    print(f" External eval every {EVAL_EVERY_STEPS} optimizer steps + end of every epoch")
    print(f" External file: {EXTERNAL_TEST_FILE}")
    print(f" Full epoch checkpoints will be written to {OUTPUT_DIR}/checkpoint_epoch_XX/")

    try:
        epoch_pbar = tqdm(range(start_epoch, EPOCHS), desc="Epochs", initial=start_epoch, total=EPOCHS)
        running_loss = 0.0

        for epoch in epoch_pbar:
            model.train()
            total_train_loss = 0
            optimizer.zero_grad()
            batch_pbar = tqdm(train_loader, desc=f"Train (Ep {epoch+1})", leave=False)

            for step, batch in enumerate(batch_pbar):
                if epoch == start_epoch and step < start_step:
                    continue

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = loss / ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

                # ---------- EXTERNAL SPLIT EVAL every EVAL_EVERY_STEPS ----------
                # Shifted left (un-indented) and changed to track raw (step + 1)
                if external_data and (step + 1) % EVAL_EVERY_STEPS == 0:
                    print(f"\n📊 [step {step + 1}] Running external split-accuracy eval...")
                    metrics = run_external_split_eval(
                        model, tokenizer, device, external_data, label_list, batch_size=32
                    )
                    if metrics:
                        msg = (
                            f"  External ({metrics['valid']} samples) | "
                            f"Line1 {metrics['line1_acc']:.1f}% | "
                            f"Line2 {metrics['line2_acc']:.1f}% | "
                            f"Full {metrics['full_acc']:.1f}%"
                        )
                        print(msg)
                        with open(DIAGNOSTIC_LOG, "a", encoding="utf-8") as f:
                            f.write(f"\n[EXTERNAL_EVAL] Epoch {epoch+1} | step {step + 1}\n")
                            f.write(msg + "\n")
                    model.train()  # back to train mode

                display_loss = loss.item() * ACCUMULATION_STEPS
                total_train_loss += display_loss
                if running_loss == 0.0:
                    running_loss = display_loss
                else:
                    running_loss = 0.9 * running_loss + 0.1 * display_loss

                if display_loss > 30.0:
                    model.eval()
                    with torch.no_grad():
                        spike_preds, _ = model(input_ids=input_ids, attention_mask=attention_mask)
                    model.train()
                    log_diagnostic(
                        tokenizer, input_ids, labels, spike_preds, id2label,
                        display_loss, epoch, step, "TRAIN_LOSS_SPIKE"
                    )

                batch_pbar.set_postfix({"loss": f"{display_loss:.4f}", "gstep": global_step})

                if (step + 1) % SAVE_STEPS == 0:
                    save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step + 1, best_val_loss, global_step)

            # end of epoch calculations
            steps_this_epoch = len(train_loader) - (start_step if epoch == start_epoch else 0)
            avg_train_loss = total_train_loss / max(1, steps_this_epoch)
            start_step = 0  # only skip on the first resumed epoch

            # ---------- VALIDATION ----------
            avg_val_loss = None
            val_em = None
            if val_loader:
                model.eval()
                total_val_loss = 0
                total_exact_match = 0
                val_samples_processed = 0
                with torch.no_grad():
                    val_pbar = tqdm(val_loader, desc=f"Val (Ep {epoch+1})", leave=False)
                    for batch in val_pbar:
                        input_ids = batch["input_ids"].to(device)
                        attention_mask = batch["attention_mask"].to(device)
                        labels = batch["labels"].to(device)
                        val_loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                        total_val_loss += val_loss.item()
                        val_preds, _ = model(input_ids=input_ids, attention_mask=attention_mask)
                        for i in range(len(val_preds)):
                            pred_seq = val_preds[i]
                            true_ids = labels[i].tolist()
                            attn = attention_mask[i].tolist()
                            true_content, pred_content = [], []
                            pred_idx = 0
                            for t_id, a in zip(true_ids, attn):
                                if a == 0:
                                    continue
                                p = pred_seq[pred_idx]
                                pred_idx += 1
                                if t_id != -100:
                                    true_content.append(t_id)
                                    pred_content.append(p)
                            if true_content == pred_content:
                                total_exact_match += 1
                            else:
                                if val_samples_processed % 400 == 0:
                                    log_diagnostic(
                                        tokenizer, input_ids[i:i+1], labels[i:i+1],
                                        [pred_seq], id2label, val_loss.item(),
                                        epoch, step, "VAL_MISMATCH_SAMPLE"
                                    )
                            val_samples_processed += 1
                avg_val_loss = total_val_loss / len(val_loader)
                val_em = total_exact_match / max(1, val_samples_processed)

            # ---------- EXTERNAL EVAL AT END OF EPOCH ----------
            ext_metrics = None
            if external_data:
                print(f"\n📊 [End of Epoch {epoch+1}] Running external split-accuracy eval...")
                ext_metrics = run_external_split_eval(
                    model, tokenizer, device, external_data, label_list, batch_size=32
                )

            # ---------- SAVE FULL EPOCH CHECKPOINT ----------
            save_epoch_checkpoint(
                model, tokenizer, optimizer, scheduler,
                epoch, best_val_loss, global_step, avg_train_loss
            )

            # ---------- BEST MODEL DECISION ----------
            was_best = False
            if avg_val_loss is not None and avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                was_best = True
                print(f"\n🌟 New best val loss ({best_val_loss:.4f}) → {OUTPUT_DIR}")
                uncompiled = getattr(model, "_orig_mod", model)
                uncompiled.save_pretrained(OUTPUT_DIR)
                tokenizer.save_pretrained(OUTPUT_DIR)

            # also keep a resume-friendly checkpoint
            save_checkpoint(model, tokenizer, optimizer, scheduler, epoch + 1, 0, best_val_loss, global_step)

            # ---------- CLEAN METRICS LOGGING ----------
            _write_epoch_metrics(
                METRICS_LOG, METRICS_JSONL,
                epoch + 1, avg_train_loss, avg_val_loss, val_em,
                ext_metrics, was_best, best_val_loss, global_step
            )

            epoch_pbar.set_postfix({
                "TrainLoss": f"{avg_train_loss:.4f}",
                "ValLoss": f"{avg_val_loss:.4f}" if avg_val_loss is not None else "n/a",
                "ValEM": f"{val_em:.1%}" if val_em is not None else "n/a",
                "ExtFull": f"{ext_metrics['full_acc']:.1f}%" if ext_metrics else "n/a"
            })

        print(f"\n✅ Training complete.")
        print(f" Best model          : {OUTPUT_DIR}")
        print(f" Per-epoch ckpts     : {OUTPUT_DIR}/checkpoint_epoch_XX/")
        print(f" Metrics report      : {METRICS_LOG}")
        print(f" Metrics (JSONL)     : {METRICS_JSONL}")

    except KeyboardInterrupt:
        print("\n\n⚠️ Training interrupted by user!")
        save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, best_val_loss, global_step)
        print("👋 Safe exit. Re-run the script to resume.")
        del model, optimizer, scheduler, train_loader
        gc.collect()
        torch.cuda.empty_cache()
        sys.exit(0)


if __name__ == "__main__":
    main()


🔍 Scanning available GPUs safely via nvidia-smi...
--> Selected GPU 1 with 17446 MB free VRAM.

📂 External test loaded: 1803 samples from data2/address_dataset.jsonl
⏳ Initialising XLM-RoBERTa-large + constrained CRF...
🔄 Loading weights from checkpoint...
⏩ Resuming from Epoch 3, Step 18 (global_step=4912)...

🚀 Training XLM-RoBERTa-large + constrained CRF...
 External eval every 2000 optimizer steps + end of every epoch
 External file: data2/address_dataset.jsonl
 Full epoch checkpoints will be written to ./xlm_roberta_large_crfV4/checkpoint_epoch_XX/


Epochs:  40%|####      | 2/5 [00:00<?, ?it/s]

Train (Ep 3):   0%|          | 0/19639 [00:00<?, ?it/s]

/home/slhui.censtatd/.local/lib/python3.12/site-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:611.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)



💾 Saving resume checkpoint at Epoch 3, Step 300 (gstep=4947)...

💾 Saving resume checkpoint at Epoch 3, Step 600 (gstep=4985)...


In [2]:
#!/usr/bin/env python3
"""
Batch Evaluation Script for XLM-RoBERTa-large + Constrained CRF
Includes Multi-Threshold Confidence Diagnostics (Baseline, 80%, 85%, 90%, 95%).
Fixed: Situation-Aware Confidence & DISTRICT tag mapping.
"""

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./xlm_roberta_large_checkpointV4"
LOG_FILE = "parsing_results_xlm_largeV3.log"
TEST_FILE = "data2/test_cleaned.jsonl"
MAX_LEN = 128
BATCH_SIZE = 32

# Note: Using 'DISTRICT' instead of 'DISTRICT' to match your training output
EXCLUDE_FROM_OVERALL = {"DISTRICT", "REGION", "SUB_DISTRICT"}
THRESHOLDS = [0.0, 0.50, 0.60, 0.70, 0.80]

# ==========================================
# CUSTOM ARCHITECTURE
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, o_id] = 0.0
        for tag in label_list:
            if tag.startswith("B-"):
                self.crf.transitions.data[o_id, label2id[tag]] = 0.0

        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O": continue
            tid = label2id[tag]

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                self.crf.transitions.data[tid, o_id] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

                self.crf.start_transitions.data[tid] = 0.0
                self.crf.end_transitions.data[tid] = 0.0

            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                self.crf.transitions.data[tid, o_id] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
                self.crf.end_transitions.data[tid] = 0.0

        self.crf.transitions.requires_grad_(False)
        self.crf.start_transitions.requires_grad_(False)
        self.crf.end_transitions.requires_grad_(False)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction='mean')
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def get_emptiest_gpu_safely():
    if not torch.cuda.is_available(): return -1
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        best_id, max_free_mb = 0, -1
        for line in result.strip().split("\n"):
            if not line.strip(): continue
            gpu_id, free_memory = map(int, line.split(", "))
            if free_memory > max_free_mb:
                max_free_mb, best_id = free_memory, gpu_id
        return best_id
    except Exception: return 0

def extract_3d_components(parsed_entities):
    components = defaultdict(list)
    confs = defaultdict(list)

    for entity in parsed_entities:
        tag = entity["entity_group"]
        if tag == "O": continue
        word = entity["word"].strip()
        if word:
            components[tag].append(word)
            confs[tag].append(entity["conf"])

    formatted_output, conf_output = {}, {}
    for tag, words in components.items():
        joined_string = "".join(words)
        if any("\u4e00" <= char <= "\u9fff" for char in joined_string):
            formatted_output[tag] = "".join(words)
        else:
            joined_en = " ".join(words).strip()
            formatted_output[tag] = re.sub(r"\s*([/\.-])\s*", r"\1", joined_en)

        conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

    return formatted_output, conf_output

def assemble_compact_json(extracted_data):
    return {
        "line1": {
            "flat": extracted_data.get("UNIT", ""), "floor": extracted_data.get("FLOOR", ""),
            "block": extracted_data.get("BLOCK", ""), "phase": extracted_data.get("PHASE", ""),
            "building_name": extracted_data.get("BUILDING_NAME", ""),
        },
        "line2": {
            "estate_name": extracted_data.get("ESTATE_NAME", ""), "village_name": extracted_data.get("VILLAGE_NAME", ""),        
            "building_number": extracted_data.get("BUILDING_NUMBER", ""), "street_name": extracted_data.get("STREET_NAME", ""),
            "sub_district": extracted_data.get("SUB_DISTRICT", ""), "district": extracted_data.get("DISTRICT", ""),
            "region": extracted_data.get("REGION", ""),
        }
    }

def flatten_json(output_dict):
    flat = {}
    if "line1" in output_dict:
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def compute_situation_aware_confidence(conf_mapped):
    core = set()
    
    # 1. Always evaluate micro-units
    if "UNIT" in conf_mapped: core.add("UNIT")
    if "FLOOR" in conf_mapped: core.add("FLOOR")
    if "BLOCK" in conf_mapped: core.add("BLOCK")
    if "PHASE" in conf_mapped: core.add("PHASE")

    # 2. Always evaluate macro structural names
    if "ESTATE_NAME" in conf_mapped: core.add("ESTATE_NAME")
    if "BUILDING_NAME" in conf_mapped: core.add("BUILDING_NAME")
    if "VILLAGE_NAME" in conf_mapped: core.add("VILLAGE_NAME")

    # 3. Always evaluate street-level routing
    if "STREET_NAME" in conf_mapped: core.add("STREET_NAME")
    if "BUILDING_NUMBER" in conf_mapped: core.add("BUILDING_NUMBER")

    overall = 1.0
    used = []
    
    for k in core:
        if k in conf_mapped and k not in EXCLUDE_FROM_OVERALL and conf_mapped[k] > 0:
            overall *= conf_mapped[k]
            used.append(k)

    # Fallback: if no core entities were found, multiply whatever is left
    if not used:
        for k, c in conf_mapped.items():
            if k not in EXCLUDE_FROM_OVERALL and c > 0:
                overall *= c
                used.append(k)

    return overall

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    device_id = get_emptiest_gpu_safely()
    device = torch.device(f"cuda:{device_id}" if device_id != -1 else "cpu")
    
    print(f"DEBUG: Loading Config, Tokenizer, and Model from {MODEL_DIR}...")
    if not os.path.exists(MODEL_DIR):
        print(f"❌ Error: {MODEL_DIR} not found.")
        return
        
    config = AutoConfig.from_pretrained(MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    
    label_list = [config.id2label[k] if isinstance(k, int) else config.id2label[str(k)] 
                  for k in sorted([int(k) for k in config.id2label.keys()])]
                  
    model = BertCRFForTokenClassification(config, MODEL_DIR, label_list)
    
    weights_path = os.path.join(MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
    
    model.to(device)
    model.eval()

    all_fields = [
        "flat", "floor", "building_name", "block", "phase",
        "estate_name", "village_name", "building_number",
        "street_name", "sub_district", "district", "region"
    ]
    
    # CRITICAL FIX: district maps to DISTRICT
    tag_map = {
        "flat": "UNIT", "floor": "FLOOR", "block": "BLOCK",
        "phase": "PHASE", "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME", "village_name": "VILLAGE_NAME",
        "building_number": "BUILDING_NUMBER", "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT", "district": "DISTRICT", "region": "REGION",
    }
    
    total_samples = 0
    total_time = 0.0
    
    stats = {t: {f: {'TP': 0, 'FP': 0, 'FN': 0} for f in all_fields} for t in THRESHOLDS}
    exact_matches = {t: 0 for t in THRESHOLDS}

    print("🚀 Running batch evaluation...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        raw_lines = [line.strip() for line in file if line.strip() and not line.startswith('#')]
    
    test_data = [json.loads(line) for line in raw_lines]
    
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Batches"):
            batch = test_data[i:i + BATCH_SIZE]
            batch_texts = [item["input"].strip() for item in batch]
            
            start_time = time.perf_counter()
            
            encoded = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=MAX_LEN, return_offsets_mapping=True, return_tensors="pt"
            )
            
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)
            offsets_batch = encoded["offset_mapping"].cpu().numpy()
            
            with torch.no_grad():
                batch_predictions, batch_emissions = model(input_ids=input_ids, attention_mask=attention_mask)
                batch_probs = torch.softmax(batch_emissions, dim=-1).cpu()

            total_time += time.perf_counter() - start_time

            for b_idx, item in enumerate(batch):
                address = batch_texts[b_idx]
                ground_truth_flat = flatten_json(item.get("output", {}))
                
                prediction_ids = batch_predictions[b_idx]
                offsets = offsets_batch[b_idx]
                item_probs = batch_probs[b_idx]
                
                char_tags = ["O"] * len(address)
                char_confs = [0.0] * len(address)

                for idx, tag_id in enumerate(prediction_ids):
                    start, end = offsets[idx]
                    if start == end: continue 

                    tag = label_list[tag_id]
                    conf = item_probs[idx, tag_id].item()

                    for c in range(start, end):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf
                            
                parsed_entities = []
                for match in re.finditer(r"[a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s]", address):
                    start_idx = match.start()
                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    
                    parsed_entities.append({
                        "entity_group": entity_group, "word": match.group(), "conf": conf,
                    })

                extracted_data, conf_output = extract_3d_components(parsed_entities)
                predicted_flat = flatten_json(assemble_compact_json(extracted_data))
                
                # Using the FIXED situation-aware confidence
                overall_conf = compute_situation_aware_confidence(conf_output)

                total_samples += 1
                
                for t in THRESHOLDS:
                    t_is_perfect_match = True
                    
                    for field in all_fields:
                        pred_val = predicted_flat.get(field, "")
                        gt_val = ground_truth_flat.get(field, "")
                        
                        tag = tag_map.get(field, field.upper())
                        field_conf = conf_output.get(tag, 0.0)
                        
                        t_pred_val = pred_val if field_conf >= t else ""
                        
                        # --- CRITICAL FIX: Case-insensitive & stripped comparison ---
                        pred_norm = t_pred_val.strip().lower()
                        gt_norm = str(gt_val).strip().lower()
                        
                        if pred_norm != gt_norm:
                            t_is_perfect_match = False
                            
                        if pred_norm and gt_norm:
                            if pred_norm == gt_norm:
                                stats[t][field]['TP'] += 1
                            else:
                                stats[t][field]['FP'] += 1
                                stats[t][field]['FN'] += 1
                        elif pred_norm and not gt_norm:
                            stats[t][field]['FP'] += 1
                        elif not pred_norm and gt_norm:
                            stats[t][field]['FN'] += 1
                    
                    if t_is_perfect_match: 
                        exact_matches[t] += 1

                log.write(f"Original: {address}\n")
                
                # Using Baseline (0.0 Threshold) for the logs
                baseline_perfect = True
                for field in all_fields:
                    pred_val = predicted_flat.get(field, "")
                    gt_val = ground_truth_flat.get(field, "")
                    
                    # Case-insensitive check for the log's EXACT MATCH status
                    if pred_val.strip().lower() != str(gt_val).strip().lower():
                        baseline_perfect = False
                        
                if baseline_perfect:
                    log.write("✅ EXACT MATCH\n")
                else:
                    log.write("❌ MISMATCH FOUND\n")
                
                for field in all_fields:
                    pred_val = predicted_flat.get(field, "")
                    gt_val = ground_truth_flat.get(field, "")
                    
                    if pred_val or gt_val:
                        # Case-insensitive check for the individual field checkmarks
                        pred_norm = pred_val.strip().lower()
                        gt_norm = str(gt_val).strip().lower()
                        status = "✅" if pred_norm == gt_norm else "❌"
                        
                        tag = tag_map.get(field, field.upper())
                        conf_str = f"  conf={conf_output.get(tag, 0.0):.4f}" if tag in conf_output else ""
                        
                        log.write(f" {status} {field.upper()}:{conf_str}\n")
                        # We still print the ORIGINAL un-lowered text so humans can read it nicely
                        log.write(f"   PRED: {pred_val if pred_val else '[None]'}\n")
                        log.write(f"   TRUE: {gt_val if gt_val else '[None]'}\n")

                log.write(f"Per-label confidences : { {k: round(v, 4) for k, v in conf_output.items()} }\n")
                log.write(f"Overall confidence    : {overall_conf:.6f}\n")
                log.write("-" * 50 + "\n")

    print("\n" + "=" * 65)
    print("📊 ADVANCED MULTI-THRESHOLD EVALUATION")
    print("=" * 65)
    print(f"Total Addresses Tested: {total_samples}")
    print(f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n")

    for t in THRESHOLDS:
        title = "BASELINE (ALL PREDICTIONS)" if t == 0.0 else f"ONLY PREDICTIONS >= {int(t*100)}% CONFIDENCE"
        print("=" * 65)
        print(f"🚀 {title}")
        print("=" * 65)
        
        if total_samples > 0:
            exact_match_acc = (exact_matches[t] / total_samples) * 100
            print(f"Whole-Address Perfect Match : {exact_match_acc:.2f}% ({exact_matches[t]}/{total_samples})\n")
            
            print(f"{'FIELD':<16} | {'PRECISION':<9} | {'RECALL':<9} | {'F1-SCORE':<9} | {'SUPPORT'}")
            print("-" * 65)
            
            macro_f1 = 0
            valid_fields = 0
            
            for field in all_fields:
                tp = stats[t][field]['TP']
                fp = stats[t][field]['FP']
                fn = stats[t][field]['FN']
                
                support = tp + fn
                
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
                
                if support > 0:
                    macro_f1 += f1
                    valid_fields += 1
                    
                p_str = f"{precision*100:>5.1f}%"
                r_str = f"{recall*100:>5.1f}%"
                f1_str = f"{f1*100:>5.1f}%"
                
                print(f"{field:<16} | {p_str:<9} | {r_str:<9} | {f1_str:<9} | {support}")

            if valid_fields > 0:
                print("-" * 65)
                print(f"{'MACRO AVERAGE':<16} | {'-':<9} | {'-':<9} | {(macro_f1/valid_fields)*100:>5.1f}%  |")
        print("\n")

    print(f"✅ Processing complete. Raw baseline logs saved to {LOG_FILE}")

if __name__ == "__main__":
    main()

DEBUG: Loading Config, Tokenizer, and Model from ./xlm_roberta_large_checkpointV3...


Some weights of XLMRobertaModel were not initialized from the model checkpoint at ./xlm_roberta_large_checkpointV3 and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bi

🚀 Running batch evaluation...


Batches:   0%|          | 0/1228 [00:00<?, ?it/s]


📊 ADVANCED MULTI-THRESHOLD EVALUATION
Total Addresses Tested: 39277
⏱️ Total Inference runtime: 217.5907 seconds

🚀 BASELINE (ALL PREDICTIONS)
Whole-Address Perfect Match : 77.77% (30545/39277)

FIELD            | PRECISION | RECALL    | F1-SCORE  | SUPPORT
-----------------------------------------------------------------
flat             |  94.2%    |  93.2%    |  93.7%    | 27627
floor            |  94.8%    |  94.4%    |  94.6%    | 34777
building_name    |  94.9%    |  94.6%    |  94.7%    | 23961
block            |  92.3%    |  93.6%    |  93.0%    | 5619
phase            |  71.1%    |  77.8%    |  74.3%    | 1049
estate_name      |  92.5%    |  92.4%    |  92.4%    | 11742
village_name     |  96.4%    |  93.8%    |  95.0%    | 10665
building_number  |  98.2%    |  98.2%    |  98.2%    | 38343
street_name      |  97.4%    |  98.4%    |  97.9%    | 28080
sub_district     |  62.7%    |  62.8%    |  62.8%    | 7744
district         |  99.7%    |  99.7%    |  99.7%    | 38862
region 

In [3]:
#!/usr/bin/env python3
"""
HK Address Parser - Final App with Integrated Evaluation (BERT-CRF Version)
Evaluates address splitting logic, calculates split-aware confidence, 
and measures Accuracy relative to Confidence Thresholds.
Switched from BiLSTM to Hugging Face BERT (XLM-RoBERTa) backend.
"""

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./xlm_roberta_large_checkpointV4"
# MODEL_DIR = "./bert_crf_modelV2"
LOG_FILE = "address_split_results_bertV3.log"
TEST_FILE = "data2/test_cleaned.jsonl"
BATCH_SIZE = 32
MAX_LEN = 128

THRESHOLDS = [0.0, 0.2, 0.3, 0.4, 0.50, 0.60, 0.70, 0.80]
ALL_FIELDS = [
    "flat", "floor", "building_name", "block", "phase",
    "estate_name", "village_name", "building_number",
    "street_name", "sub_district", "district", "region"
]

# ==========================================
# UTILITY FUNCTIONS FOR EVALUATION
# ==========================================
def flatten_json(output_dict):
    """Flattens the nested ground truth JSON from test.jsonl."""
    flat = {}
    if "line1" in output_dict and isinstance(output_dict["line1"], dict):
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def normalize_for_eval(text):
    """Lowercases and removes ALL spaces/punctuation for strict string comparison."""
    if not text:
        return ""
    return re.sub(r'[\s,/\\\-;\.，。、；]+', '', str(text).lower())


# ==========================================
# CUSTOM BERT-CRF ARCHITECTURE
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, o_id] = 0.0
        for tag in label_list:
            if tag.startswith("B-"):
                self.crf.transitions.data[o_id, label2id[tag]] = 0.0

        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O": continue
            tid = label2id[tag]

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                self.crf.transitions.data[tid, o_id] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

                self.crf.start_transitions.data[tid] = 0.0
                self.crf.end_transitions.data[tid] = 0.0

            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                self.crf.transitions.data[tid, o_id] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
                self.crf.end_transitions.data[tid] = 0.0

        self.crf.transitions.requires_grad_(False)
        self.crf.start_transitions.requires_grad_(False)
        self.crf.end_transitions.requires_grad_(False)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction='mean')
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions


# ==========================================
# ADDRESS PARSER CLASS (BERT Version)
# ==========================================
class HKAddressParserBERT:
    def __init__(self, model_path):
        self.model_path = model_path
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")

        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model path not found: {model_path}.")

        # Load Tokenizer & Config
        self.config = AutoConfig.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        # Ensure label_list is sorted consistently by string/int IDs
        self.label_list = [self.config.id2label[k] if isinstance(k, int) else self.config.id2label[str(k)] 
                           for k in sorted([int(k) for k in self.config.id2label.keys()])]

        # Initialize Model
        self.model = BertCRFForTokenClassification(self.config, model_path, self.label_list)
        
        weights_path = os.path.join(model_path, "pytorch_model.bin")
        if not os.path.exists(weights_path):
            raise FileNotFoundError(f"Model weights (pytorch_model.bin) not found in {model_path}.")
            
        print(f"📦 Loading weights from {weights_path} ...")
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available(): return torch.device("cpu")
        try:
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id, max_free_mb = 0, 0
            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id, free_memory, gpu_util = int(parts[0]), int(parts[1]), int(parts[2])
                if gpu_util < 30 and free_memory > max_free_mb:
                    max_free_mb, best_id = free_memory, gpu_id
            return torch.device(f"cuda:{best_id}")
        except Exception: return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)

        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O": continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])

        formatted_output, conf_output = {}, {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities, conf_mapped):
        """
        Splits address and calculates confidence strictly based on labels 
        that influenced the split decision. Includes logic to correctly bind 
        Building Numbers with Village and Street names, and separates Estate/Phase 
        from Building/Block when both hierarchical levels exist.
        """
        line1_keys = set()
        logic_keys_used = set()
        
        # 1. Micro elements
        if "flat" in extracted_labels: 
            line1_keys.add("flat")
            logic_keys_used.add("flat")
        if "floor" in extracted_labels: 
            line1_keys.add("floor")
            logic_keys_used.add("floor")

        # 2. Structural elements (Hierarchical separation logic)
        has_bldg = "building_name" in extracted_labels
        has_block = "block" in extracted_labels
        has_est = "estate_name" in extracted_labels
        has_phase = "phase" in extracted_labels
        has_vill = "village_name" in extracted_labels
        has_street = "street_name" in extracted_labels
        has_bldg_no = "building_number" in extracted_labels
        
        # RULE 1: If Building OR Block exists, they take priority for Line 1 (micro).
        if has_bldg or has_block:
            if has_bldg:
                line1_keys.add("building_name")
                logic_keys_used.add("building_name")
            if has_block:
                line1_keys.add("block")
                logic_keys_used.add("block")
                
        # RULE 2: If NO Building or Block, but Estate exists, Estate acts as the building (Line 1).
        elif has_est:
            line1_keys.add("estate_name")
            logic_keys_used.add("estate_name")
            if has_phase: 
                line1_keys.add("phase")
                logic_keys_used.add("phase")
                
        # RULE 3: Fallbacks for Village / Street / Building Number
        elif has_vill:
            line1_keys.add("village_name")
            logic_keys_used.add("village_name")
            if has_bldg_no: 
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_street:
            line1_keys.add("street_name")
            logic_keys_used.add("street_name")
            if has_bldg_no: 
                line1_keys.add("building_number")
                logic_keys_used.add("building_number")
        elif has_bldg_no:
            line1_keys.add("building_number")
            logic_keys_used.add("building_number")

        # 3. Assign tokens
        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        token_groups = []
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        
        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag in line1_keys: token_groups.append("micro")
            elif mapped_tag != "o" and mapped_tag != "O": token_groups.append("macro")
            else: token_groups.append("O") 
                
        # Resolve punctuation
        resolved_groups = []
        last_valid = "micro" if not is_chinese else "macro" 
        for tg in token_groups:
            if tg != "O":
                last_valid = tg
                resolved_groups.append(tg)
            else:
                resolved_groups.append(last_valid)
                
        # Group adjacent tokens into segments by their tag
        micro_segments, macro_segments = [], []
        curr_micro_seg, curr_micro_tag = [], None
        curr_macro_seg, curr_macro_tag = [], None
        
        for entity, group in zip(parsed_entities, resolved_groups):
            tag = mapping.get(entity["entity_group"], entity["entity_group"].lower())
            word = entity["word"]
            
            if group == "micro":
                if tag != 'o' and tag != 'O':
                    if curr_micro_tag != tag:
                        if curr_micro_seg:
                            micro_segments.append((curr_micro_tag, curr_micro_seg))
                        curr_micro_seg = [word]
                        curr_micro_tag = tag
                    else:
                        curr_micro_seg.append(word)
                else:
                    if curr_micro_seg: curr_micro_seg.append(word)
                    else: curr_micro_seg, curr_micro_tag = [word], "o"
            else:
                if tag != 'o' and tag != 'O':
                    if curr_macro_tag != tag:
                        if curr_macro_seg:
                            macro_segments.append((curr_macro_tag, curr_macro_seg))
                        curr_macro_seg = [word]
                        curr_macro_tag = tag
                    else:
                        curr_macro_seg.append(word)
                else:
                    if curr_macro_seg: curr_macro_seg.append(word)
                    else: curr_macro_seg, curr_macro_tag = [word], "o"
                        
        if curr_micro_seg: micro_segments.append((curr_micro_tag, curr_micro_seg))
        if curr_macro_seg: macro_segments.append((curr_macro_tag, curr_macro_seg))
            
        # ---------------------------------------------------------
        # REMOVED reorder_segments() - Strings now stay exactly
        # in the order the user typed them!
        # def reorder_segments(segments, is_chinese):
        #     """Moves the building_number chunk to sit directly next to the primary grouping element"""
        #     bldg_no_idx = next((i for i, s in enumerate(segments) if s[0] == "building_number"), -1)
        #     if bldg_no_idx == -1: return segments
            
        #     bldg_no_seg = segments.pop(bldg_no_idx)
        #     target_tags = ["village_name", "street_name", "estate_name", "building_name"]
            
        #     if is_chinese:
        #         target_idx = -1
        #         for i, s in enumerate(segments):
        #             if s[0] in target_tags: target_idx = i
        #         if target_idx != -1: segments.insert(target_idx + 1, bldg_no_seg)
        #         else: segments.insert(0, bldg_no_seg)
        #     else:
        #         target_idx = -1
        #         for i, s in enumerate(segments):
        #             if s[0] in target_tags:
        #                 target_idx = i
        #                 break
        #         if target_idx != -1: segments.insert(target_idx, bldg_no_seg)
        #         else: segments.append(bldg_no_seg)
        #     return segments

        # micro_segments = reorder_segments(micro_segments, is_chinese)
        # macro_segments = reorder_segments(macro_segments, is_chinese)
        # ---------------------------------------------------------

        micro_string = "".join("".join(words) for tag, words in micro_segments)
        macro_string = "".join("".join(words) for tag, words in macro_segments)
        
        def clean_string(s):
            s = re.sub(r"^[,/\\\-;\s，。、；]+", "", s)
            s = re.sub(r"[,/\\\-;\s，。、；]+$", "", s)
            return re.sub(r"\s{2,}", " ", s).strip()

        micro_string = clean_string(micro_string)
        macro_string = clean_string(macro_string)
        
        # Calculate Logic Confidence
        split_conf = 1.0
        for k in logic_keys_used:
            split_conf *= conf_mapped.get(k, 1.0)
            
        if not logic_keys_used:
            for k, c in conf_mapped.items():
                if k not in ["district", "region", "sub_district"]: split_conf *= c

        line1 = macro_string if is_chinese else micro_string
        line2 = micro_string if is_chinese else macro_string

        return line1, line2, split_conf, list(logic_keys_used)

    def parse_batch(self, full_addresses, batch_size=32):
        all_results = []
        for i in tqdm(range(0, len(full_addresses), batch_size), desc="Processing"):
            batch = full_addresses[i : i + batch_size]
            
            # 1. Tokenize using HF tokenizer with offsets mapping
            encoded = self.tokenizer(
                batch, padding=True, truncation=True, max_length=MAX_LEN,
                return_offsets_mapping=True, return_tensors="pt"
            )
            
            input_ids = encoded["input_ids"].to(self.device)
            attention_mask = encoded["attention_mask"].to(self.device)
            offsets_batch = encoded["offset_mapping"].cpu().numpy()

            # 2. Get predictions and emissons map
            with torch.no_grad():
                batch_predictions, batch_emissions = self.model(input_ids=input_ids, attention_mask=attention_mask)
                batch_probs = torch.softmax(batch_emissions, dim=-1).cpu()

            for idx, address_str in enumerate(batch):
                prediction_ids = batch_predictions[idx]
                offsets = offsets_batch[idx]
                item_probs = batch_probs[idx]
                
                # 3. Map tokens back to the original string character by character (BERT Subword to Char mapping)
                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)
                
                for j, tag_id in enumerate(prediction_ids):
                    start, end = offsets[j]
                    if start == end: continue # Skip special tokens (e.g., [CLS], [SEP])
                    
                    tag = self.label_list[tag_id]
                    conf = item_probs[j, tag_id].item()

                    for c in range(start, end):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf

                # 4. Extract with trailing spaces using regex to preserve original string formatting exactly
                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z]+|[0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)

                    tag = char_tags[start_idx] if start_idx < len(char_tags) else "O"
                    conf = char_confs[start_idx] if start_idx < len(char_confs) else 0.0
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"

                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })

                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)
                
                # 5. Split and get logic-aware confidence
                line1, line2, split_conf, used_keys = self._split_address(
                    extracted_mapped, address_str, parsed_entities, conf_mapped
                )

                all_results.append((address_str, extracted_mapped, conf_mapped, line1, line2, split_conf, used_keys))

        return all_results

# ==========================================
# MAIN EXECUTION & EVALUATION LOOP
# ==========================================
def main():
    if not os.path.exists(MODEL_DIR) or not os.path.exists(TEST_FILE):
        print("❌ Error: Model directory or Test file not found.")
        return

    parser = HKAddressParserBERT(model_path=MODEL_DIR)

    print(f"🚀 Loading dataset from {TEST_FILE}...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        test_data = [json.loads(line) for line in file if line.strip() and not line.startswith("#")]
        
    inputs = [item["input"].strip() for item in test_data]
    
    start_time = time.perf_counter()
    results = parser.parse_batch(inputs, batch_size=BATCH_SIZE)
    total_time = time.perf_counter() - start_time

    # Initialize Metrics Structure
    stats = {
        t: {
            "total_in_bin": 0,
            "logic_correct": 0,
            "line1_correct": 0,
            "line2_correct": 0,
            "full_correct": 0
        } for t in THRESHOLDS
    }

    excluded_count = 0

    print(f"✍️ Evaluating and writing results to {LOG_FILE}...")
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for idx, (address, pred_tags, pred_confs, line1, line2, split_conf, used_keys) in enumerate(results):
            ground_truth_flat = flatten_json(test_data[idx].get("output", {}))
            
            # --- CHECK FOR CORRUPTED GROUND TRUTH ---
            norm_address = normalize_for_eval(address)
            is_corrupted = False
            for _, gt_val in ground_truth_flat.items():
                norm_gt_val = normalize_for_eval(gt_val)
                if norm_gt_val and norm_gt_val not in norm_address:
                    is_corrupted = True
                    break
            
            if is_corrupted:
                excluded_count += 1
                log.write(f"--- Result {idx + 1} [EXCLUDED: CORRUPTED DATA] ---\n")
                log.write(f"Original Input  : {address}\n")
                log.write("Reason          : Ground truth contains values not present in the input text.\n")
                log.write("-" * 50 + "\n")
                continue

            # --- EVALUATE CORRECTNESS OF THE PREDICTION ---
            pred_ff = normalize_for_eval(pred_tags.get("floor", "") + pred_tags.get("flat", ""))
            gt_ff = normalize_for_eval(ground_truth_flat.get("floor", "") + ground_truth_flat.get("flat", ""))
            
            p_bldg = normalize_for_eval(pred_tags.get("building_name", ""))
            p_est  = normalize_for_eval(pred_tags.get("estate_name", ""))
            g_bldg = normalize_for_eval(ground_truth_flat.get("building_name", ""))
            g_est  = normalize_for_eval(ground_truth_flat.get("estate_name", ""))
            
            field_correct = {}
            for field in ALL_FIELDS:
                p_norm = normalize_for_eval(pred_tags.get(field, ""))
                g_norm = normalize_for_eval(ground_truth_flat.get(field, ""))
                
                if p_norm == g_norm:
                    field_correct[field] = True
                elif field in ['floor', 'flat'] and pred_ff == gt_ff and pred_ff != "":
                    field_correct[field] = True
                elif field in ['building_name', 'estate_name'] and (p_bldg == g_est and p_est == g_bldg):
                    field_correct[field] = True
                else:
                    field_correct[field] = False
                    
            # 1. Split Logic Determination Correctness
            is_logic_correct = all(field_correct.get(k, False) for k in used_keys) if used_keys else True
            
            # 2. Line 1 Exact Match (Micro fields)
            line1_fields = ["flat", "floor", "block", "phase", "building_name", "estate_name"]
            is_l1_correct = all(field_correct[f] for f in line1_fields)
            
            # 3. Line 2 Exact Match (Macro fields)
            line2_fields = ["village_name", "building_number", "street_name", "sub_district", "district", "region"]
            is_l2_correct = all(field_correct[f] for f in line2_fields)
            
            # 4. Full Address Exact Match
            is_full_correct = is_l1_correct and is_l2_correct

            # --- BIN THE ACCURACY BY CONFIDENCE ---
            for t in THRESHOLDS:
                if split_conf >= t:
                    stats[t]["total_in_bin"] += 1
                    if is_logic_correct: stats[t]["logic_correct"] += 1
                    if is_l1_correct: stats[t]["line1_correct"] += 1
                    if is_l2_correct: stats[t]["line2_correct"] += 1
                    if is_full_correct: stats[t]["full_correct"] += 1

            # --- WRITE FULL LOGS ---
            log.write(f"--- Result {idx + 1} ---\n")
            log.write(f"Original Input  : {address}\n")
            log.write(f"Logic Keys Used : {used_keys}\n")
            log.write(f"Split Conf      : {split_conf:.6f} (Product of keys used in split)\n")
            log.write(f"Output Line 1   : {line1}\n")
            log.write(f"Output Line 2   : {line2}\n")
            
            if is_full_correct: log.write("✅ EXACT MATCH\n")
            else: log.write("❌ MISMATCH FOUND\n")
            
            for field in ALL_FIELDS:
                pred_val = pred_tags.get(field, "")
                gt_val = ground_truth_flat.get(field, "")
                if pred_val or gt_val:
                    status = "✅" if field_correct[field] else "❌"
                    conf_str = f"  conf={pred_confs.get(field, 0.0):.4f}"
                    log.write(f" {status} {field.upper()}:{conf_str}\n")
                    log.write(f"   PRED: {pred_val if pred_val else '[None]'}\n")
                    log.write(f"   TRUE: {gt_val if gt_val else '[None]'}\n")
                    
            log.write("-" * 50 + "\n")

        # --- WRITE SUMMARY TABLE ---
        total_samples = len(inputs)
        valid_samples = total_samples - excluded_count
        
        def build_table_str():
            out = "\n" + "=" * 65 + "\n"
            out += "📊 ADDRESS SPLITTING METRICS (BERT-CRF CALIBRATED)\n"
            out += "=" * 65 + "\n"
            out += f"Total Addresses Provided: {total_samples}\n"
            out += f"Excluded (Corrupted)    : {excluded_count}\n"
            out += f"Total Valid Evaluated   : {valid_samples}\n"
            out += f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n\n"

            for t in THRESHOLDS:
                title = "ALL PREDICTIONS (Threshold 0%)" if t == 0.0 else f"PREDICTIONS WITH ≥ {int(t*100)}% CONFIDENCE"
                out += "-" * 65 + "\n"
                out += f"🚀 {title}\n"
                out += "-" * 65 + "\n"
                
                total_in_bin = stats[t]["total_in_bin"]
                if total_in_bin > 0:
                    out += f"{'METRIC':<35} | {'ACCURACY (Correct / Total in Bin)'}\n"
                    out += "-" * 65 + "\n"
                    
                    l_acc = (stats[t]["logic_correct"] / total_in_bin) * 100
                    l1_acc = (stats[t]["line1_correct"] / total_in_bin) * 100
                    l2_acc = (stats[t]["line2_correct"] / total_in_bin) * 100
                    full_acc = (stats[t]["full_correct"] / total_in_bin) * 100
                    
                    out += f"{'Split Logic Determination Correct':<35} | {l_acc:>6.2f}%  ({stats[t]['logic_correct']}/{total_in_bin})\n"
                    out += f"{'Line 1 (Micro) Components Correct':<35} | {l1_acc:>6.2f}%  ({stats[t]['line1_correct']}/{total_in_bin})\n"
                    out += f"{'Line 2 (Macro) Components Correct':<35} | {l2_acc:>6.2f}%  ({stats[t]['line2_correct']}/{total_in_bin})\n"
                    out += f"{'Full Address Perfect Match':<35} | {full_acc:>6.2f}%  ({stats[t]['full_correct']}/{total_in_bin})\n\n"
                else:
                    out += f"No samples met the >= {int(t*100)}% confidence threshold.\n\n"
            return out

        table_output = build_table_str()
        print(table_output)
        log.write(table_output)

if __name__ == "__main__":
    main()

DEBUG: Using Device -> cuda:0


Some weights of XLMRobertaModel were not initialized from the model checkpoint at ./xlm_roberta_large_checkpointV3 and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bi

📦 Loading weights from ./xlm_roberta_large_checkpointV3/pytorch_model.bin ...
🚀 Loading dataset from data2/test.jsonl...


Processing:   0%|          | 0/1228 [00:00<?, ?it/s]

✍️ Evaluating and writing results to address_split_results_bertV3.log...

📊 ADDRESS SPLITTING METRICS (BERT-CRF CALIBRATED)
Total Addresses Provided: 39277
Excluded (Corrupted)    : 0
Total Valid Evaluated   : 39277
⏱️ Total Inference runtime: 229.0188 seconds

-----------------------------------------------------------------
🚀 ALL PREDICTIONS (Threshold 0%)
-----------------------------------------------------------------
METRIC                              | ACCURACY (Correct / Total in Bin)
-----------------------------------------------------------------
Split Logic Determination Correct   |  98.49%  (38682/39277)
Line 1 (Micro) Components Correct   |  98.78%  (38796/39277)
Line 2 (Macro) Components Correct   |  98.98%  (38877/39277)
Full Address Perfect Match          |  97.94%  (38468/39277)

-----------------------------------------------------------------
🚀 PREDICTIONS WITH ≥ 20% CONFIDENCE
-----------------------------------------------------------------
METRIC                